# Breast Cancer Detection Project

Steps to take:

Phase 1 : Data

1. Get the data
2. EDA first - Check and Analyse
    - Feature Transformation:
    - Standardization, Scale, or PCA
3. Feature Selection
4. Train test split

Phase 2: Model Building

5. Modelling
6. Testing

Phase 3: Deployment

7. Export the model
8. Check for important features

---------
9. We write a function that takes those important values (4 top important ones), and makes a prediction on those, and returns the prediction
10. We will deploy the model on Gradio on Hugging Face
11. We will learn deployment with Streamlit as well

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
df = pd.DataFrame(load_breast_cancer().data, columns= load_breast_cancer().feature_names)
df['target'] = load_breast_cancer().target
# df['target'] = df['target'].map({0: 'Malignant', 1: 'Benign'})

display(df.sample(5))
display(df.shape)
display(df.isnull().sum())
display(df.describe())
df.info()

#### Let's find the most impotant features and use them only for this project

- we are checking the correlation matrix
- each feature will be compared with other features
- whichever features have high correlation values, will be kept
- or the other way is, to check correlation ONLY with the Target Column

In [ ]:
corr_matrix = df.corr()

plt.figure(figsize= (20, 15))
sns.heatmap(df.corr(), annot= True, cmap= 'coolwarm', fmt= '.2f')
plt.show()

In [ ]:
corr_matrix = df.corr()['target']
corr_matrix

In [ ]:
corr_matrix['target'] = 0
threshold = 0.7

high_corr_target_cols = [col for col in corr_matrix.columns if corr_matrix[col].max() > threshold]
print(len(high_corr_target_cols))
high_corr_target_cols

In [ ]:
np.fill_diagonal(corr_matrix.values, 0)

threshold = 0.9
high_corr_cols = [col for col in corr_matrix.columns if corr_matrix[col].max() > threshold]
print(len(high_corr_cols))
high_corr_cols

In [ ]:
x = load_breast_cancer().data

pca_model = PCA(n_components= 1)

principal_component = pca_model.fit_transform(x)
principal_component.round(2)

In [ ]:
X = load_breast_cancer().data
y = load_breast_cancer().target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.2, random_state= 42)

model_rf = RandomForestClassifier(n_estimators= 10)

model_rf.fit(X_train, y_train)

In [ ]:
importances_rf = model_rf.feature_importances_
df_feature_importance_rf = pd.DataFrame({
    'features': load_breast_cancer().feature_names,
    'importance': importances_rf
})
df_top_features_rf = df_feature_importance_rf.sort_values(by= 'importance', ascending= False).head(4)
df_top_features_rf

### Saving an ML model

- When we train a model, we can save the trained model and use it somewhere else as well
- 
- What these files look like
- Using:
    - Pickle .pkl
    - Joblib .joblib
    - .keras, .h5 - TensorFlow way of saving - Framework by Google
    - onnx - opensource representation - a way to convert models around
    - .pt .pth - PyTorch way of saving - Framework by Meta/Facebook

**Pickle** is a standard Python library for serializing and deserializing Python objects, including ML models. It is easy to use but may not be secure for untrusted data

In [ ]:
import pickle

with open('Breast Cancer Random Forest.pkl', 'wb') as file:
    pickle.dump(model_rf, file)

with open('Breast Cancer Random Forest.pkl', 'rb') as file:
    model = pickle.load(file)

model

**Joblib** is optimized for saving large NumPy arrays and is more efficient than Pickle for large models.

In [ ]:
import joblib

joblib.dump(model_rf, 'Breast Cancer Random Forest.joblib')

model_joblib = joblib.load('Breast Cancer Random Forest.joblib')

model_joblib

**JSON** You can save the model architecture as a JSON file, which is useful for custom models where you need full control over the restoration process.